# 🚀 Predilex NLP — Entraînement sur Google Colab

Ce notebook entraîne les deux modèles CamemBERT sur GPU Colab.

**Avant de commencer :**
1. `Runtime` → `Change runtime type` → **GPU (T4 ou A100)**
2. Avoir le projet sur GitHub **ET** les données sur Google Drive

**Structure Google Drive attendue :**
```
Mon Drive/
└── predilex_data/
    ├── Y_train_predilex.csv
    ├── x_train_ids.csv
    └── txt_files/
        ├── Agen_100515.txt
        └── ...
```

**Durée estimée :**
- GPU T4  : ~1h (sexe ~15 min + dates ~45 min)
- GPU A100: ~20 min

---
## Étape 1 — Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU détecté : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("❌ Pas de GPU — change le runtime : Runtime > Change runtime type > GPU")
    raise SystemExit("GPU requis")

---
## Étape 2 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive monté dans /content/drive/")

---
## Étape 3 — Cloner le repo GitHub

In [ ]:
GITHUB_URL = "https://github.com/djema-rayane/predilex-nlp.git"

import os

if os.path.exists("/content/predilex-nlp"):
    print("Repo déjà cloné — mise à jour...")
    os.chdir("/content/predilex-nlp")
    !git pull
else:
    !git clone {GITHUB_URL} /content/predilex-nlp
    os.chdir("/content/predilex-nlp")

print(f"✅ Répertoire de travail : {os.getcwd()}")
!ls

---
## Étape 4 — Installation des dépendances

In [ ]:
!pip install -q transformers[torch] accelerate sentencepiece datasets pyyaml scikit-learn
print("✅ Dépendances installées")

# Vérification des versions
import transformers, torch, accelerate
print(f"  torch        : {torch.__version__}")
print(f"  transformers : {transformers.__version__}")
print(f"  accelerate   : {accelerate.__version__}")

---
## Étape 5 — Lier les données depuis Google Drive

In [ ]:
import os, shutil
from pathlib import Path

# ⚠️ VÉRIFIE que ce chemin correspond à ton Drive
DRIVE_DATA = "/content/drive/MyDrive/predilex_data"

# Vérification
required = [
    f"{DRIVE_DATA}/Y_train_predilex.csv",
    f"{DRIVE_DATA}/x_train_ids.csv",
    f"{DRIVE_DATA}/txt_files",
]
for path in required:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {path}")
    if not exists:
        raise FileNotFoundError(
            f"Fichier manquant : {path}\n"
            f"Upload tes données dans Drive/predilex_data/"
        )

# Créer les dossiers dans le projet
Path("/content/predilex-nlp/data/raw").mkdir(parents=True, exist_ok=True)
Path("/content/predilex-nlp/data/processed").mkdir(parents=True, exist_ok=True)
Path("/content/predilex-nlp/models").mkdir(parents=True, exist_ok=True)

# Créer des liens symboliques (évite de copier 770 fichiers)
RAW = "/content/predilex-nlp/data/raw"

for fname in ["Y_train_predilex.csv", "x_train_ids.csv"]:
    dst = f"{RAW}/{fname}"
    if not os.path.exists(dst):
        os.symlink(f"{DRIVE_DATA}/{fname}", dst)

txt_dst = f"{RAW}/txt_files"
if not os.path.exists(txt_dst):
    os.symlink(f"{DRIVE_DATA}/txt_files", txt_dst)

# Vérification finale
n_txt = len(list(Path(txt_dst).glob("*.txt")))
print(f"\n✅ Données liées : {n_txt} fichiers .txt")

---
## Étape 6 — Vérification du chargement des données

In [ ]:
import sys
sys.path.insert(0, '/content/predilex-nlp')
os.chdir('/content/predilex-nlp')

from src.data_loader import load_config, load_train_data

cfg = load_config('configs/config.yaml')
df  = load_train_data(cfg)

print(f"\n✅ {len(df)} documents chargés")
print(f"Sexe      : {df['sexe'].value_counts().to_dict()}")
print(f"Text moy  : {df['text'].str.len().mean():.0f} chars")

---
## Étape 7 — Entraînement du modèle SEXE

Fine-tuning CamemBERT pour prédire homme/femme.
- **Input** : premiers 512 tokens du document
- **Classes** : 2 (homme=0, femme=1)
- **Durée** : ~15 min sur T4

In [ ]:
!python src/train_sex.py --config configs/config.yaml

In [ ]:
from pathlib import Path
sex_model_path = Path('models/sex_classifier/best_model')
if sex_model_path.exists():
    files = list(sex_model_path.iterdir())
    print(f"✅ Modèle sexe sauvegardé : {len(files)} fichiers")
    for f in files:
        size = f.stat().st_size / 1e6
        print(f"   {f.name:<35} {size:.1f} MB")
else:
    print("❌ Modèle introuvable — vérifier les logs ci-dessus")

---
## Étape 8 — Entraînement du modèle DATES

Fine-tuning CamemBERT pour classifier les phrases de dates.
- **Input** : phrase + contexte ±2 (256 tokens)
- **Classes** : 3 (date_accident=0, date_consolidation=1, autre_date=2)
- **Durée** : ~45 min sur T4

In [ ]:
!python src/train_dates.py --config configs/config.yaml

In [ ]:
date_model_path = Path('models/date_classifier/best_model')
if date_model_path.exists():
    files = list(date_model_path.iterdir())
    print(f"✅ Modèle dates sauvegardé : {len(files)} fichiers")
    for f in files:
        size = f.stat().st_size / 1e6
        print(f"   {f.name:<35} {size:.1f} MB")
else:
    print("❌ Modèle introuvable — vérifier les logs ci-dessus")

---
## Étape 9 — Sauvegarde des modèles sur Google Drive

> ⚠️ **IMPORTANT** : les modèles sont stockés dans la session Colab (volatile).
> Si la session se ferme, ils sont perdus. On les copie sur Drive maintenant.

In [ ]:
import shutil

DRIVE_MODELS = "/content/drive/MyDrive/predilex_models"
os.makedirs(DRIVE_MODELS, exist_ok=True)

for model_name in ["sex_classifier", "date_classifier"]:
    src = f"/content/predilex-nlp/models/{model_name}/best_model"
    dst = f"{DRIVE_MODELS}/{model_name}"

    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        size_mb = sum(f.stat().st_size for f in Path(dst).rglob('*') if f.is_file()) / 1e6
        print(f"✅ {model_name} sauvegardé sur Drive ({size_mb:.0f} MB)")
    else:
        print(f"❌ {model_name} introuvable")

print(f"\nModèles disponibles dans : {DRIVE_MODELS}")

---
## Étape 10 — Inférence sur le train set (vérification)

In [ ]:
# Vérification : prédire sur le train set pour voir les métriques globales
!python src/predict.py --train-set

---
## Étape 11 — Inférence sur le test set (soumission)

> À lancer uniquement quand le test set Predilex est disponible.

In [ ]:
# Lier le test set depuis Drive (quand disponible)
# DRIVE_TEST = "/content/drive/MyDrive/predilex_data/test"
# os.symlink(f"{DRIVE_TEST}/x_test_ids.csv", "data/raw/x_test_ids.csv")
# os.symlink(f"{DRIVE_TEST}/txt_files", "data/raw/x_test_txt_files")

# !python src/predict.py

print("Décommenter les lignes ci-dessus quand le test set est disponible.")

---
## Étape 12 — Télécharger la soumission

In [ ]:
from google.colab import files
import os

submission_path = "data/processed/submission.csv"

if os.path.exists(submission_path):
    files.download(submission_path)
    print(f"✅ Soumission téléchargée")
else:
    print("Pas encore de fichier de soumission — lance l'étape 11 d'abord.")
    
    # Télécharger la vérification sur train set si disponible
    check_path = "data/processed/submission_train_check.csv"
    if os.path.exists(check_path):
        files.download(check_path)
        print(f"✅ Vérification train téléchargée")